<div style="border-left:6px solid #14b8a6;padding:18px 22px;background:#f7f7f7">
<p style="margin:0;color:#14b8a6;letter-spacing:2px;font-size:12px">DATA MINING E INTELIGENCIA DE NEGOCIOS · GUÍA 3 · MÉTODOS NO SUPERVISADOS</p>
<h1 style="margin:6px 0 4px 0;font-weight:300">Guía 3 en Google Colab: segmentación de clientes con K-Means</h1>
<p style="margin:0;font-size:16px;color:#444">Plantilla de trabajo del grupo. Caso Mercado Central.</p>
<p style="margin:10px 0 0 0;color:#777;font-size:13px">Profe Alejandra Otero · Universidad EAN · 2026</p>
</div>

<a href="https://colab.research.google.com/github/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/blob/main/data-mining-36993/notebooks/Guia3_KMeans_Plantilla_Colab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

Este cuaderno **reemplaza el trabajo en SPSS y en Excel** de la Guía 3. Aquí van a hacer todo el recorrido: elegir variables, correr el K-Means, justificar el número de clústeres, perfilar los grupos con variables activas y pasivas, y llegar a la infografía.

### Datos del grupo

| Campo | Completar |
|---|---|
| Número de grupo en Canvas | Grupo __ |
| Integrantes (nombre completo) | 1. <br>2. <br>3. <br>4. |
| Herramienta de IA | Gemini en Google Colab |
| Enlace de este cuaderno compartido como lector | (peguen aquí el enlace, ver el paso final) |

## Antes de empezar: cómo se trabaja este cuaderno

**1. Guarden su propia copia.** Este cuaderno está en modo lectura. Vayan a **Archivo > Guardar una copia en Drive**. Esa copia es la del grupo: compártanla entre ustedes con el botón **Compartir** (arriba a la derecha) como **Editor** para trabajar juntos.

**2. Los datos ya vienen listos.** La primera celda de código carga la **base Mercado Central ya depurada** (la misma del taller de limpieza de la Guía 1 y de la sesión de K-Means) directamente desde GitHub. No tienen que subir ni limpiar nada: todos los grupos trabajan con la misma base limpia.

**3. Ustedes escriben el código con ayuda de Gemini, la IA que ya viene integrada en Colab.** Es gratuita y no necesitan ninguna otra herramienta. Para abrirla, usen el botón de **Gemini** (icono de chispa) en la parte superior de Colab, o el enlace **generar** que aparece en las celdas de código vacías.

Cada paso tiene esta estructura:

| Bloque | Qué hacen ustedes |
|---|---|
| **Qué tienen que lograr** | Leerlo con calma: es lo que Gemini debe ayudarles a construir |
| **Pistas técnicas** | Nombres de funciones y librerías que les sirven para pedirle a Gemini algo preciso |
| **Cómo saber que quedó bien** | Revisar el resultado antes de seguir |
| **Celda de código vacía** | Pegar el código, ejecutarlo con el botón de reproducir (o Ctrl + Enter) y corregir hasta que funcione |
| **Respuesta del grupo** | Interpretar el resultado con sus palabras |

**4. El prompt lo escriben ustedes.** No hay instrucciones prefabricadas para Gemini: parte del aprendizaje es saber pedir. Un buen prompt le cuenta a Gemini **qué datos tienen** (nombre de la tabla `df` y de las columnas), **qué quieren lograr** y **en qué herramienta están** (Python en Google Colab). Si el código falla, copien el error completo y pídanle que lo corrija. Si no entienden una línea, pídanle que se la explique.

**5. Ejecuten en orden, de arriba hacia abajo.** Si cierran Colab o se desconecta, vayan a **Entorno de ejecución > Ejecutar todas**.

**6. Documenten su uso de Gemini.** Al final hay una bitácora donde copian los prompts principales que usaron y qué tuvieron que corregir. Esa bitácora hace parte de la entrega.

> **Material de referencia:** la guía interactiva y el cuaderno de la Sesión 5 (en Canvas, módulo *Recursos de presencia permanente*), la grabación de la tutoría y la lectura de la guía: Gironés, J., Casas, J., Minguillón, J. y Caihuelas, R. (2017). *Minería de datos: modelos y algoritmos*. Editorial UOC. Capítulo 8.

## Paso 0. Cargar la base (esta celda ya está lista: solo ejecútenla)

Importa las librerías básicas y lee la base depurada desde GitHub. Al ejecutarla deben ver: **2.030 clientes y 41 variables**, y **0 valores faltantes**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL_DATOS = "https://raw.githubusercontent.com/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/main/data-mining-36993/data/mercado_central_depurada.csv"
df = pd.read_csv(URL_DATOS)

print(f"La base tiene {df.shape[0]:,} clientes (filas) y {df.shape[1]} variables (columnas)")
print("Valores faltantes en toda la base:", df.isna().sum().sum())
df.head()

### Diccionario de variables principales

| Variable | Qué significa | Tipo |
|---|---|---|
| `id_cliente` | Identificador del cliente (no se usa en el modelo) | Identificador |
| `edad`, `anio_nacimiento` | Edad del cliente en años y año de nacimiento | Numérica |
| `nivel_educativo` | Básico, Pregrado, Maestría, Doctorado | Categórica |
| `estado_civil`, `situacion_pareja`, `vive_en_pareja` | Estado civil; si vive en pareja o solo (texto y 0/1) | Categórica |
| `ingreso_anual_hogar` | Ingreso anual del hogar | Numérica |
| `ninos_en_casa`, `adolescentes_en_casa`, `hijos_total` | Número de niños, adolescentes y total de hijos en el hogar | Numérica |
| `dias_desde_ultima_compra` | Recencia: días desde la última compra (menos es más reciente) | Numérica |
| `gasto_vinos`, `gasto_frutas`, `gasto_carnes`, `gasto_pescados`, `gasto_dulces`, `gasto_productos_premium` | Gasto por categoría en los últimos 2 años | Numérica |
| `gasto_total` | Suma del gasto en todas las categorías | Numérica |
| `compras_web`, `compras_catalogo`, `compras_tienda`, `compras_total` | Número de compras por canal y en total | Numérica |
| `compras_con_descuento` | Compras hechas con descuento | Numérica |
| `visitas_web_mes` | Visitas al sitio web en el último mes | Numérica |
| `acepto_campana_1` a `acepto_campana_5`, `campanas_aceptadas` | Si aceptó cada campaña anterior (0/1) y cuántas aceptó | Binaria / numérica |
| `respondio_ultima_campana` | Si respondió a la última campaña (0/1) | Binaria |
| `reclamo_ultimos_2_anios` | Si puso un reclamo en los últimos 2 años (0/1) | Binaria |
| `antiguedad_dias` | Días desde que se registró como cliente | Numérica |
| `ingreso_imputado`, `anio_imputado`, `atipico_*` | Marcas del proceso de limpieza de la Guía 1 (no se usan en el modelo) | Control |

## Paso 1. Pregunta de negocio y objetivos ajustados

Este paso corresponde al **punto 1 de la guía**: con base en la retroalimentación de la Guía 2, ajusten sus objetivos. No necesita código.

**Qué tienen que lograr:** dejar escrita la pregunta de negocio que va a responder la segmentación y los objetivos del análisis, ya corregidos.

**Cómo saber que quedó bien:** la pregunta empieza por algo como *¿Qué tipos de clientes...?* y cada objetivo específico se puede verificar con una tabla o gráfica de este cuaderno.

**Respuesta del grupo.**

- **Pregunta de negocio:**
- **Objetivo general:**
- **Objetivos específicos (2 o 3):**
  1.
  2.
  3.
- **Qué cambiamos frente a la Guía 2 y por qué (según la retroalimentación):**

## Paso 2. Conocer las variables candidatas

**Qué tienen que lograr:** revisar las variables numéricas que podrían entrar al modelo: su promedio, desviación estándar, mínimo y máximo. Así ven que están en escalas muy distintas (miles en ingreso, unidades en hijos).

**Pistas técnicas:** `df.describe()`, `df.select_dtypes()`, seleccionar columnas con `df[["col1", "col2"]]`.

**Cómo saber que quedó bien:** obtienen una tabla con una fila por variable (o una columna por variable) con sus estadísticos básicos.

In [ ]:
# PASO 2: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** ¿Qué variables tienen escalas más grandes? ¿Por qué eso sería un problema para un método que mide distancias?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 3. Elegir variables activas y pasivas

Este es el concepto central de la guía (**punto 5**).

| | Variables **activas** | Variables **pasivas** |
|---|---|---|
| ¿Entran al K-Means? | Sí: con ellas se forman los grupos | No |
| ¿Para qué sirven? | Definen la similitud entre clientes | Describen y validan los grupos después |
| ¿De qué tipo? | Numéricas | Pueden ser categóricas o binarias |
| ¿Cómo se analizan? | Promedios por clúster | Tablas de contingencia y chi-cuadrado |

**Qué tienen que lograr:**

1. Elegir **entre 4 y 8 variables activas** numéricas que respondan a su pregunta de negocio.
2. Elegir **al menos 3 variables pasivas** (por ejemplo, nivel educativo, situación de pareja, respuesta a la última campaña, rangos de edad).
3. Guardarlas en dos listas de Python, `activas` y `pasivas`, y crear una tabla `X` solo con las activas.

**Pistas técnicas:** listas de Python (`activas = [...]`), `X = df[activas]`. No incluyan `id_cliente` ni las marcas de limpieza (`atipico_*`, `*_imputado`).

**Cómo saber que quedó bien:** `X` tiene 2.030 filas y tantas columnas como variables activas eligieron.

In [ ]:
# PASO 3: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** Justifiquen cada variable.

| Variable | Activa o pasiva | Por qué la elegimos (relación con la pregunta de negocio) |
|---|---|---|
| | | |
| | | |
| | | |

## Paso 4. Estandarizar las variables activas

**Qué tienen que lograr:** transformar las variables activas a puntajes z (promedio 0 y desviación estándar 1) para que ninguna pese más solo por estar en una escala mayor. Guarden el resultado en `X_z`.

**Pistas técnicas:** `StandardScaler` de `sklearn.preprocessing`, método `fit_transform()`.

**Cómo saber que quedó bien:** al revisar `X_z`, cada columna tiene promedio aproximado de 0 y desviación aproximada de 1.

In [ ]:
# PASO 4: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** En una frase: ¿qué pasaría con los grupos si no estandarizaran?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 5. ¿Cuántos clústeres? Criterio de elección de k

Este paso corresponde al **punto 4 de la guía** y al criterio de rúbrica *Justificación del criterio de elección de los clústeres*.

**Qué tienen que lograr:**

1. Correr el K-Means para **k de 2 a 8**.
2. Para cada k, calcular **al menos dos criterios**: la **inercia** (para el método del codo) y el **coeficiente de silueta**. Si quieren ir más allá, agreguen Calinski-Harabasz (más alto es mejor) y Davies-Bouldin (más bajo es mejor).
3. Mostrar los resultados en una **tabla** y en una **gráfica** (codo y silueta).

**Pistas técnicas:** `KMeans` de `sklearn.cluster` con `n_init=10` y `random_state=42` (así todos obtienen resultados reproducibles), atributo `.inertia_`, `silhouette_score` de `sklearn.metrics`, un ciclo `for k in range(2, 9)`, `matplotlib` para graficar.

**Cómo saber que quedó bien:** la inercia siempre baja al aumentar k; la silueta sube y baja. La tabla tiene 7 filas (k = 2 a 8).

In [ ]:
# PASO 5: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.**

- **Número de clústeres elegido (k):**
- **Criterio estadístico** (qué muestran el codo y la silueta para ese k, con los valores):
- **Criterio de negocio** (por qué ese número de grupos es útil y accionable para Mercado Central):
- **Por qué no elegimos un k mayor o menor:**

## Paso 6. Entrenar el modelo final y la proporción de cada grupo

Corresponde al **punto 4 de la guía**: *detallar el número de clústeres elegidos, bajo qué criterio y la proporción de los grupos*.

**Qué tienen que lograr:**

1. Entrenar el K-Means con el k que eligieron (`n_init=10`, `random_state=42`).
2. Agregar a `df` una columna `cluster` con el grupo de cada cliente (numerado desde 1).
3. Mostrar una tabla con **cuántos clientes** y **qué porcentaje** del total hay en cada clúster.

**Pistas técnicas:** `fit_predict()`, `value_counts()`, `normalize=True`.

**Cómo saber que quedó bien:** la suma de clientes da 2.030 y la suma de porcentajes da 100 %.

In [ ]:
# PASO 6: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** ¿Los grupos quedaron de tamaños parecidos o hay uno mucho más grande? ¿Qué significa eso para el negocio?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 7. Perfil de los clústeres con las variables activas

Corresponde a los **puntos 5 y 6 de la guía** (análisis de variables activas y perfiles).

**Qué tienen que lograr:**

1. Una tabla con el **promedio de cada variable activa por clúster**, en sus unidades originales (para que se entienda: pesos, días, número de compras).
2. Una **visualización** que permita ver rápido en qué se diferencia cada grupo del promedio: por ejemplo, un mapa de calor de los puntajes z promedio por clúster, o barras con un índice (100 = promedio general).

**Pistas técnicas:** `df.groupby("cluster")[activas].mean()`, `imshow` o `seaborn.heatmap` para el mapa de calor. Pidan a Gemini un título que diga la conclusión, no solo el nombre de la gráfica.

**Cómo saber que quedó bien:** pueden decir, para cada clúster, en qué 2 o 3 variables está claramente por encima o por debajo del promedio.

In [ ]:
# PASO 7: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** Para cada clúster: ¿qué variables activas lo distinguen (por encima o por debajo del promedio)?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 8. Tablas de contingencia con las variables pasivas

Corresponde al **punto 5 de la guía**: *análisis de las variables activas y pasivas usando tablas de contingencia*. Esto reemplaza el Excel de la versión anterior.

**Qué tienen que lograr:** para **cada variable pasiva** (mínimo 3):

1. Una tabla de contingencia **clúster x variable pasiva** en **conteos**.
2. La misma tabla en **porcentaje por fila** (de cada clúster, qué % está en cada categoría). Esta es la que se interpreta, porque los clústeres tienen tamaños distintos.
3. La **prueba chi-cuadrado**: si el p-valor es menor que 0,05, la variable pasiva está asociada con los clústeres y sirve para describirlos.

Si una pasiva es numérica (por ejemplo, `edad`), conviértanla primero en rangos.

**Pistas técnicas:** `pd.crosstab()` con `normalize="index"`, `chi2_contingency` de `scipy.stats`, `pd.cut()` para crear rangos.

**Cómo saber que quedó bien:** cada fila de la tabla en porcentaje suma 100 %, y para cada pasiva tienen un p-valor.

In [ ]:
# PASO 8: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.**

| Variable pasiva | p-valor | ¿Asociación significativa? | Qué nos dice de los clústeres |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

## Paso 9. Perfiles y nombres de los segmentos

Corresponde al **punto 6 de la guía**: *determinar los perfiles de características de cada clúster*. Es la base de la infografía.

**Qué tienen que lograr:** para cada clúster, un perfil completo que combine lo que vieron en el paso 7 (activas) y en el paso 8 (pasivas), con un nombre que describa al grupo y una acción para el negocio.

Un truco para redactar: *"Son clientes que ___, ___ y ___; por eso Mercado Central debería ___"*.

**Cómo saber que quedó bien:** cada afirmación del perfil se puede señalar en una tabla o gráfica de este cuaderno.

**Respuesta del grupo.**

| Clúster | Nombre del segmento | % de clientes | Qué los distingue (activas) | Quiénes son (pasivas) | Acción recomendada | Métrica para medirla |
|---|---|---|---|---|---|---|
| 1 | | | | | | |
| 2 | | | | | | |
| 3 | | | | | | |

## Paso 10. Gráficas para la infografía

Corresponde al **punto 7 de la guía**: *realizar una infografía donde muestre los diferentes perfiles de los clústeres*.

**Qué tienen que lograr:** generar y descargar como imagen (PNG) **al menos dos gráficas** que van a usar en la infografía. Por ejemplo: el tamaño de cada segmento, el perfil de cada segmento frente al promedio, o la variable pasiva más reveladora (como la tasa de respuesta a campañas por segmento).

**Pistas técnicas:** `plt.savefig("nombre.png", dpi=200, bbox_inches="tight")` y, en Colab, `from google.colab import files` y `files.download("nombre.png")`.

**Recomendaciones de diseño:** un color de acento para lo que quieren destacar y grises para el resto, título con la conclusión, ejes con nombre y la fuente al pie (*Fuente: elaboración propia con la base Mercado Central*).

**Cómo saber que quedó bien:** los archivos PNG aparecen en sus descargas (o en el panel de archivos de Colab, icono de carpeta a la izquierda).

In [ ]:
# PASO 10: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

## Paso 11. Conclusiones y recomendación

**Qué tienen que lograr:** responder la pregunta de negocio del paso 1 con base en los resultados.

**Respuesta del grupo.**

1. **Respuesta a la pregunta de negocio:**
2. **Tres hallazgos principales (con el dato que los respalda):**
   -
   -
   -
3. **Recomendación prioritaria para Mercado Central y por qué:**
4. **Limitaciones del análisis:**

## Paso 12. Bitácora de uso de Gemini

**Qué tienen que lograr:** dejar constancia de cómo trabajaron con Gemini. No se califica cuánto la usaron, sino que la hayan usado con criterio.

**Respuesta del grupo.**

| Paso | Prompt principal que le escribimos a Gemini (cópienlo tal cual) | ¿Funcionó a la primera? Qué tuvimos que corregir o preguntar |
|---|---|---|
| 2 | | |
| 5 | | |
| 8 | | |
| | | |

**Lo que aprendimos al pedirle código a Gemini (2 o 3 líneas):**

## Antes de entregar: lista de chequeo

- [ ] Ejecutamos **Entorno de ejecución > Reiniciar y ejecutar todas** y todo corre sin errores, de arriba hacia abajo.
- [ ] Todas las celdas de **Respuesta del grupo** están diligenciadas.
- [ ] Llenamos los datos del grupo al inicio (integrantes y herramienta de IA).
- [ ] Compartimos el cuaderno: **Compartir > Acceso general > Cualquier persona con el enlace > Lector**, y pegamos el enlace en la tabla del inicio.
- [ ] Descargamos el cuaderno: **Archivo > Descargar > Descargar .ipynb**.
- [ ] Hicimos la **infografía** de perfiles (PDF o imagen) con las gráficas del paso 10.

### Qué se sube a Canvas (tarea *Guía 3*, una sola entrega por grupo)

1. **El cuaderno de Colab** (archivo `.ipynb` con todas las celdas ejecutadas y el enlace compartido escrito al inicio).
2. **La infografía** de perfiles de los clústeres (PDF o PNG).

Usen la opción **Cargar archivo** y adjunten los dos archivos en la misma entrega.

<p style="color:#777;font-size:13px">Dudas: en la tutoría del jueves 29 de octubre a las 6:00 p. m. las resolvemos todas, paso por paso. También pueden escribirme por el Inbox de Canvas. Profe Alejandra Otero.</p>